# Breast Cancer Detection - Google Colab Demo
## Multi-Objective Optimization with NSGA-III

This notebook demonstrates the complete pipeline for breast cancer detection under dataset shift.

**Steps:**
1. Setup environment and install dependencies
2. Mount Google Drive and load datasets
3. Test preprocessing pipeline
4. Test model training
5. Run NSGA-III optimization (small scale)
6. Zero-shot evaluation on INbreast

## 1. Environment Setup

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install dependencies
!pip install -q pydicom opencv-python-headless scikit-image
!pip install -q pymoo
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118

In [ ]:
# Import core libraries
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Mount Google Drive and Setup Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Upload the breast_cancer_detection folder to your Google Drive
# Then update this path to point to it

PROJECT_PATH = "/content/drive/MyDrive/breast_cancer_detection"

# Verify the path exists
if os.path.exists(PROJECT_PATH):
    print(f"✓ Project found at: {PROJECT_PATH}")
    # Add to Python path
    sys.path.insert(0, PROJECT_PATH)
else:
    print(f"✗ Project not found at: {PROJECT_PATH}")
    print("Please upload the breast_cancer_detection folder to your Google Drive")

In [ ]:
# Configure data paths
# UPDATE THESE PATHS to match your Google Drive structure

VINDR_IMAGES_ROOT = "/content/drive/MyDrive/vindr-mammo/images"
VINDR_CSV = "/content/drive/MyDrive/vindr-mammo/metadata/stratified_selection.csv"

INBREAST_DICOM_DIR = "/content/drive/MyDrive/INbreast/AllDICOMs"
INBREAST_CSV = "/content/drive/MyDrive/INbreast/INbreast.csv"

# Verify paths
print("Checking data paths...")
print(f"VinDr images: {os.path.exists(VINDR_IMAGES_ROOT)}")
print(f"VinDr CSV: {os.path.exists(VINDR_CSV)}")
print(f"INbreast DICOM: {os.path.exists(INBREAST_DICOM_DIR)}")
print(f"INbreast CSV: {os.path.exists(INBREAST_CSV)}")

In [ ]:
# ============================================================================
# EXPERIMENT CONFIGURATION
# ============================================================================
#
# Configure the entire notebook for DEMO or PRODUCTION mode
# Change EXPERIMENT_MODE to switch between modes
#
# ============================================================================

from datetime import datetime

# Set experiment mode: "DEMO" or "PRODUCTION"
EXPERIMENT_MODE = "DEMO"  # Change to "PRODUCTION" for main experiment

print("="*80)
print(f"EXPERIMENT MODE: {EXPERIMENT_MODE}")
print("="*80)

if EXPERIMENT_MODE == "PRODUCTION":
    # ========================================================================
    # PRODUCTION CONFIGURATION (20-50 hours)
    # ========================================================================
    
    print("\n[PRODUCTION MODE]")
    print("  - Uses FULL VinDr-Mammo dataset")
    print("  - NSGA-III: 20 population x 50 generations = 1000 evaluations")
    print("  - Training: 50 epochs per model")
    print("  - Adaptive preprocessing: ENABLED")
    print("  - Checkpointing: ENABLED")
    print("  - Expected runtime: 20-50 hours")
    print("  - GPU required: Tesla T4 or better")
    
    # Global configuration
    CONFIG = {
        # Dataset settings
        'use_full_dataset': True,
        'train_batch_size': 8,
        'val_batch_size': 8,
        'num_workers': 2,
        
        # Preprocessing
        'use_adaptive_preprocessing': True,
        
        # Training settings
        'max_epochs': 50,
        'patience': 10,
        'learning_rate': 1e-4,
        'weight_decay': 1e-4,
        
        # NSGA-III settings
        'nsga3_population': 20,
        'nsga3_generations': 50,
        'enable_checkpointing': True,
        'resume_if_available': True,
        
        # Run ID (auto-generated with timestamp)
        'run_id': f"production_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
        
        # Mode
        'mode': 'PRODUCTION'
    }
    
    print(f"\nRun ID: {CONFIG['run_id']}")
    print("\n[WARNING] This will take 20-50 hours!")
    print("[WARNING] Ensure GPU runtime is selected")
    print("[WARNING] Ensure Google Drive has 20+ GB free space")
    print("[WARNING] For Colab free tier: Plan for disconnections")
    print("           Checkpoints will save progress automatically.")
    
elif EXPERIMENT_MODE == "DEMO":
    # ========================================================================
    # DEMO CONFIGURATION (1-2 hours)
    # ========================================================================
    
    print("\n[DEMO MODE]")
    print("  - Uses SUBSET of VinDr-Mammo (~100 samples)")
    print("  - NSGA-III: 5 population x 3 generations = 15 evaluations")
    print("  - Training: 5 epochs per model")
    print("  - Adaptive preprocessing: ENABLED")
    print("  - Checkpointing: ENABLED")
    print("  - Expected runtime: 1-2 hours")
    print("  - Good for testing and development")
    
    # Global configuration
    CONFIG = {
        # Dataset settings
        'use_full_dataset': False,
        'train_batch_size': 4,
        'val_batch_size': 4,
        'num_workers': 2,
        
        # Preprocessing
        'use_adaptive_preprocessing': True,
        
        # Training settings
        'max_epochs': 5,
        'patience': 3,
        'learning_rate': 1e-4,
        'weight_decay': 1e-4,
        
        # NSGA-III settings
        'nsga3_population': 5,
        'nsga3_generations': 3,
        'enable_checkpointing': True,
        'resume_if_available': True,
        
        # Run ID
        'run_id': f"demo_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
        
        # Mode
        'mode': 'DEMO'
    }
    
    print(f"\nRun ID: {CONFIG['run_id']}")
    print("\n[INFO] This is a quick test run")
    print("[INFO] Results will not be production-quality")
    print("[INFO] Use this to verify the pipeline works end-to-end")
    
else:
    raise ValueError(f"Invalid EXPERIMENT_MODE: {EXPERIMENT_MODE}. Must be 'DEMO' or 'PRODUCTION'")

# Display configuration
print("\n" + "="*80)
print("CONFIGURATION SUMMARY")
print("="*80)
print(f"\nDataset:")
print(f"  Full dataset: {CONFIG['use_full_dataset']}")
print(f"  Batch size: {CONFIG['train_batch_size']}")
print(f"\nTraining:")
print(f"  Max epochs: {CONFIG['max_epochs']}")
print(f"  Patience: {CONFIG['patience']}")
print(f"  Learning rate: {CONFIG['learning_rate']}")
print(f"\nOptimization:")
print(f"  Population: {CONFIG['nsga3_population']}")
print(f"  Generations: {CONFIG['nsga3_generations']}")
print(f"  Total evals: {CONFIG['nsga3_population'] * CONFIG['nsga3_generations']}")
print(f"\nOther:")
print(f"  Adaptive preprocessing: {CONFIG['use_adaptive_preprocessing']}")
print(f"  Checkpointing: {CONFIG['enable_checkpointing']}")
print(f"  Resume enabled: {CONFIG['resume_if_available']}")
print(f"  Run ID: {CONFIG['run_id']}")
print("\n" + "="*80)

# Set individual variables for backward compatibility
USE_FULL_DATASET = CONFIG['use_full_dataset']
USE_ADAPTIVE_PREPROCESSING = CONFIG['use_adaptive_preprocessing']
RUN_ID = CONFIG['run_id']
ENABLE_CHECKPOINTING = CONFIG['enable_checkpointing']
RESUME_IF_AVAILABLE = CONFIG['resume_if_available']

print("\n[OK] Configuration loaded successfully")
print("[OK] All sections will use these settings automatically\n")

## 3. Preprocessing Configuration

**Choose preprocessing mode:**
- **Adaptive** = Entropy-based domain adaptation (mitigates domain shift)
- **Standard** = Baseline preprocessing (original implementation)

Set `USE_ADAPTIVE_PREPROCESSING = True` below to enable adaptation.

In [ ]:
# ============================================================================
# PREPROCESSING CONFIGURATION
# ============================================================================

# Set this to True to enable adaptive preprocessing with entropy adaptation
# Set to False for baseline (standard) preprocessing
# USE_ADAPTIVE_PREPROCESSING is now set by CONFIG above
# USE_ADAPTIVE_PREPROCESSING = True

print("="*60)
print("PREPROCESSING CONFIGURATION")
print("="*60)

# Initialize global variables
entropy_stats = None
ENTROPY_STATS_AVAILABLE = False

if USE_ADAPTIVE_PREPROCESSING:
    print("\n[SELECTED] Adaptive Preprocessing with Entropy Adaptation")
    print("  - Adjusts image contrast to match training distribution")
    print("  - Mitigates domain shift for zero-shot evaluation")
    print("  - Requires entropy cache (computed once from training set)")
    
    # Check if entropy cache exists
    from pathlib import Path
    cache_path = Path(PROJECT_PATH) / "cache" / "entropy_stats_vindr_train.json"
    
    if cache_path.exists():
        print(f"\n  [OK] Entropy cache found at: {cache_path}")
        try:
            from src.domain_adaptation import EntropyStatistics
            entropy_stats = EntropyStatistics.load(str(cache_path))
            print(f"      Mean: {entropy_stats.mean:.4f} bits")
            print(f"      Std:  {entropy_stats.std:.4f} bits")
            ENTROPY_STATS_AVAILABLE = True
        except Exception as e:
            print(f"  [ERROR] Failed to load entropy stats: {e}")
            print(f"      Will run without adaptation")
            entropy_stats = None
            ENTROPY_STATS_AVAILABLE = False
    else:
        print(f"\n  [WARNING] Entropy cache not found")
        print(f"      Expected: {cache_path}")
        print(f"      Will run without adaptation (same as baseline)")
        print(f"      To enable: Run Section 16 to compute entropy stats")
        ENTROPY_STATS_AVAILABLE = False
        entropy_stats = None
else:
    print("\n[SELECTED] Standard Preprocessing (Baseline)")
    print("  - No entropy adaptation")
    print("  - Matches original implementation")
    entropy_stats = None
    ENTROPY_STATS_AVAILABLE = False

print("\n" + "="*60)

## 4. Test Preprocessing Pipeline

Test the configured preprocessor (adaptive or standard) on a sample DICOM image.

In [ ]:
# Create preprocessor based on configuration
if USE_ADAPTIVE_PREPROCESSING:
    from src.adaptive_preprocessing import AdaptiveMammographyPreprocessor
    
    preprocessor = AdaptiveMammographyPreprocessor(
        target_size=(720, 480),
        aspect_ratio=1.5,
        entropy_stats=entropy_stats if ENTROPY_STATS_AVAILABLE else None,
        apply_adaptation=ENTROPY_STATS_AVAILABLE  # Only adapt if cache exists
    )
    
    print(f"[OK] AdaptiveMammographyPreprocessor created")
    print(f"     Adaptation: {'ENABLED' if ENTROPY_STATS_AVAILABLE else 'DISABLED (no cache)'}")
else:
    from src.preprocessing import MammographyPreprocessor
    
    preprocessor = MammographyPreprocessor()
    print("[OK] MammographyPreprocessor created (baseline)")

In [ ]:
# Test on a sample DICOM file
# Replace with an actual path from your dataset
sample_dicom = "/content/drive/MyDrive/vindr-mammo/images/STUDY_ID/IMAGE_ID.dicom"

if os.path.exists(sample_dicom):
    processed_img = preprocessor(sample_dicom)
    
    print(f"Processed image shape: {processed_img.shape}")
    print(f"Expected shape: (480, 720, 3)")
    print(f"Value range: [{processed_img.min()}, {processed_img.max()}]")
    
    # Show adaptation metrics if using adaptive preprocessing
    if USE_ADAPTIVE_PREPROCESSING and hasattr(preprocessor, 'get_adaptation_summary'):
        summary = preprocessor.get_adaptation_summary()
        if summary:
            print(f"\nAdaptation Metrics:")
            print(f"  Mean initial entropy: {summary['mean_initial_entropy']:.4f} bits")
            print(f"  Mean final entropy: {summary['mean_final_entropy']:.4f} bits")
            print(f"  Mean iterations: {summary['mean_iterations']:.1f}")
            print(f"  Convergence rate: {summary['convergence_rate']*100:.1f}%")
    
    # Visualize
    plt.figure(figsize=(8, 6))
    plt.imshow(processed_img)
    
    if USE_ADAPTIVE_PREPROCESSING:
        plt.title("Preprocessed Mammogram (Adaptive)")
    else:
        plt.title("Preprocessed Mammogram (Standard)")
    
    plt.axis('off')
    plt.show()
else:
    print(f"Sample file not found. Please update the path.")

## 5. Test Dataset Loading

Load VinDr-Mammo dataset with the configured preprocessor.

In [ ]:
from src.datasets import VinDRMammoBinaryDataset, create_breast_level_splits

# Load dataset with configured preprocessor
print("Loading VinDr-Mammo dataset...")
print(f"Preprocessor: {'Adaptive' if USE_ADAPTIVE_PREPROCESSING else 'Standard'}")

dataset = VinDRMammoBinaryDataset(
    images_root=VINDR_IMAGES_ROOT,
    csv_file=VINDR_CSV,
    preprocessor=preprocessor
)

print(f"\n[OK] Dataset loaded: {len(dataset)} samples")

# Get dataset statistics
labels = []
for i in range(len(dataset)):
    _, label = dataset[i]
    labels.append(label.item())

n_benign = sum([1 for l in labels if l == 0])
n_malignant = sum([1 for l in labels if l == 1])

print(f"\nDataset Statistics:")
print(f"  Benign: {n_benign}")
print(f"  Malignant: {n_malignant}")
print(f"  Class ratio: {n_benign/n_malignant:.2f}:1")

## 6. Test Model Building

In [ ]:
# Test loading a sample
img, label = dataset[0]

print(f"\nSample data:")
print(f"  Image shape: {img.shape}")
print(f"  Image dtype: {img.dtype}")
print(f"  Value range: [{img.min():.3f}, {img.max():.3f}]")
print(f"  Label: {label.item()} ({'Malignant' if label.item() == 1 else 'Benign'})")

# Visualize
plt.figure(figsize=(8, 6))
plt.imshow(img.permute(1, 2, 0))
plt.title(f"Sample Image - {'Malignant' if label.item() == 1 else 'Benign'}")
plt.axis('off')
plt.show()

## 7. Test Augmentation

In [ ]:
from src.models import build_resnet152

# Build model with different configurations
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Testing model configurations...\n")

# Test 1: Full fine-tuning
model1 = build_resnet152(
    pretrained=True,
    dropout=0.2,
    unfreeze_fraction=1.0
)
info1 = model1.get_trainable_params_info()
print("Config 1 - Full fine-tuning (unfreeze=1.0):")
print(f"  Total params: {info1['total_params']:,}")
print(f"  Trainable: {info1['trainable_params']:,} ({info1['trainable_percentage']:.1f}%)\n")

# Test 2: Partial fine-tuning
model2 = build_resnet152(
    pretrained=True,
    dropout=0.2,
    unfreeze_fraction=0.5
)
info2 = model2.get_trainable_params_info()
print("Config 2 - Partial fine-tuning (unfreeze=0.5):")
print(f"  Total params: {info2['total_params']:,}")
print(f"  Trainable: {info2['trainable_params']:,} ({info2['trainable_percentage']:.1f}%)\n")

# Test 3: Feature extraction only
model3 = build_resnet152(
    pretrained=True,
    dropout=0.2,
    unfreeze_fraction=0.0
)
info3 = model3.get_trainable_params_info()
print("Config 3 - Feature extraction (unfreeze=0.0):")
print(f"  Total params: {info3['total_params']:,}")
print(f"  Trainable: {info3['trainable_params']:,} ({info3['trainable_percentage']:.1f}%)")

# Test forward pass
model1 = model1.to(device)
test_input = torch.randn(2, 3, 480, 720).to(device)
output = model1(test_input)
print(f"\n✓ Forward pass successful: Input {test_input.shape} → Output {output.shape}")

## 8. Test Training Pipeline (Quick Demo)

In [ ]:
from src.augmentations import get_augmentation

# Test different augmentation strengths
img_sample = dataset[0][0]  # Get first image

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

strengths = [0.0, 0.3, 0.6, 1.0]

for i, strength in enumerate(strengths):
    aug = get_augmentation(strength)
    img_aug = aug(img_sample.clone())
    
    axes[i].imshow(img_aug.permute(1, 2, 0))
    axes[i].set_title(f"Augmentation Strength: {strength}")
    axes[i].axis('off')

plt.tight_layout()
plt.show()

print("✓ Augmentation test completed")

In [ ]:
# ============================================================================
# DATASET SELECTION: DEMO MODE vs FULL DATASET
# ============================================================================

# Set this to True to use the FULL dataset for training/validation
# Set to False to use a small subset for quick testing (demo mode)
# USE_FULL_DATASET is now set by CONFIG above
# USE_FULL_DATASET = False

print("="*60)
print("DATASET SELECTION")
print("="*60)

if USE_FULL_DATASET:
    print("\n[SELECTED] Full Dataset Mode")
    print("  - Uses entire VinDr-Mammo dataset")
    print("  - 80/20 train/validation split")
    print("  - Training will take significantly longer")
    print("  - Recommended for production runs")
else:
    print("\n[SELECTED] Demo Mode (Small Subset)")
    print("  - Uses ~100 training samples, ~30 validation samples")
    print("  - Fast training for testing the pipeline")
    print("  - Not recommended for production")

print("\n" + "="*60)

In [ ]:
from src.training import train_model
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
import numpy as np

print("Setting up training pipeline...\n")

if USE_FULL_DATASET:
    # Full dataset mode: Use proper train/val split
    print("Using FULL dataset with stratified 80/20 split")
    
    # Get all labels for stratification
    all_labels = [dataset[i][1].item() for i in range(len(dataset))]
    all_indices = np.arange(len(dataset))
    
    # Stratified train/val split
    train_indices, val_indices = train_test_split(
        all_indices,
        test_size=0.2,
        stratify=all_labels,
        random_state=42
    )
    
    train_dataset = Subset(dataset, train_indices)
    val_dataset = Subset(dataset, val_indices)
    
    print(f"  Training set: {len(train_dataset)} samples")
    print(f"  Validation set: {len(val_dataset)} samples")
    
    # Use larger batch size for full dataset
    batch_size = 8
    max_epochs = 50
    
else:
    # Demo mode: Use small subset for quick testing
    print("Using DEMO subset for quick testing")
    
    n_samples = len(dataset)
    n_train_demo = min(100, int(n_samples * 0.8))
    n_val_demo = min(30, int(n_samples * 0.2))
    
    train_indices = np.random.choice(n_samples, n_train_demo, replace=False)
    val_indices = np.random.choice(
        [i for i in range(n_samples) if i not in train_indices],
        n_val_demo,
        replace=False
    )
    
    train_dataset = Subset(dataset, train_indices)
    val_dataset = Subset(dataset, val_indices)
    
    print(f"  Demo training set: {len(train_dataset)} samples")
    print(f"  Demo validation set: {len(val_dataset)} samples")
    
    # Use smaller batch size and fewer epochs for demo
    batch_size = 4
    max_epochs = 5

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"\n[OK] Data loaders created")
print(f"  Batch size: {batch_size}")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Max epochs: {max_epochs}")

# Test batch loading
imgs, labels = next(iter(train_loader))
print(f"\nBatch test:")
print(f"  Image batch shape: {imgs.shape}")
print(f"  Label batch: {labels}")

# Initialize model for training
from src.models import build_resnet152

model = build_resnet152(
    pretrained=True,
    dropout=0.2,
    unfreeze_fraction=0.3  # Fine-tune last 30% of backbone
)

model = model.to(device)
print(f"\n[OK] Model initialized and moved to {device}")

# Show trainable parameters
info = model.get_trainable_params_info()
print(f"  Total params: {info['total_params']:,}")
print(f"  Trainable: {info['trainable_params']:,} ({info['trainable_percentage']:.1f}%)")

In [ ]:
# Train the model
print("\n" + "="*80)
print(f"TRAINING MODEL ({'FULL DATASET' if USE_FULL_DATASET else 'DEMO MODE'})")
print("="*80 + "\n")

model, metrics = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    val_dataset=val_dataset,
    device=device,
    learning_rate=1e-4,
    weight_decay=1e-4,
    patience=5,
    max_epochs=max_epochs,  # Use variable from above
    pos_weight=n_benign/n_malignant,
    verbose=True
)

print("\n" + "="*80)
print("TRAINING COMPLETED!")
print("="*80)
print(f"\nFinal Metrics:")
print(f"  PR-AUC:               {metrics['pr_auc']:.4f}")
print(f"  AUROC:                {metrics['auroc']:.4f}")
print(f"  Brier Score:          {metrics['brier']:.4f}")
print(f"  Robustness Degrad.:   {metrics['robustness_degradation']:.4f}")
print("="*80)

## 9. Test Breast-Level Aggregation (Noisy-OR)

In [ ]:
# Test on real dataset
print("\nTesting breast-level aggregation on validation set...\n")

y_true_breast, y_probs_breast = aggregate_breast_level_predictions(
    model, val_dataset, device
)

print(f"Number of breasts evaluated: {len(y_true_breast)}")
print(f"Benign breasts: {sum(y_true_breast == 0)}")
print(f"Malignant breasts: {sum(y_true_breast == 1)}")
print(f"\nBreast-level probability range: [{y_probs_breast.min():.3f}, {y_probs_breast.max():.3f}]")

## 10. Test Robustness Evaluation

In [ ]:
from src.robustness import RobustnessTester

print("Testing robustness to intensity perturbations...\n")

# Create robustness tester
tester = RobustnessTester(
    brightness_delta=0.1,
    contrast_factor=0.1,
    noise_std=0.02
)

# Evaluate robustness on validation set
results = tester.evaluate_robustness(model, val_loader, device)

print(f"\nRobustness Evaluation Results:")
print(f"  PR-AUC (clean):       {results['pr_auc_standard']:.4f}")
print(f"  PR-AUC (perturbed):   {results['pr_auc_perturbed']:.4f}")
print(f"  Degradation:          {results['degradation']:.4f}")

print(f"\nInterpretation:")
if results['degradation'] < 0.05:
    print(f"  [EXCELLENT] Model is highly robust to intensity perturbations")
elif results['degradation'] < 0.10:
    print(f"  [GOOD] Model shows good robustness")
elif results['degradation'] < 0.15:
    print(f"  [MODERATE] Model has moderate robustness")
else:
    print(f"  [POOR] Model is sensitive to perturbations")

print(f"\nNote: Lower degradation = more robust model")

## 11. NSGA-III Optimization (Small Scale Demo)

**Warning:** Full optimization with 20 population × 50 generations = 1000 evaluations will take 20-50 hours.

This demo runs a **small-scale** version (5 population × 3 generations = 15 evaluations) for demonstration.

In [ ]:
from src.optimization import BreastCancerOptimizationProblem
from pymoo.algorithms.moo.nsga3 import NSGA3
from pymoo.optimize import minimize
from pymoo.util.ref_dirs import get_reference_directions
import pickle
import os
from datetime import datetime
import time

print("="*80)
print("NSGA-III OPTIMIZATION (DEMO VERSION)")
print("="*80)
print("\nWarning: Full optimization takes 20-50 hours!")
print("This demo uses: 5 population x 3 generations = 15 evaluations")
print("For production, use: 20 population x 50 generations = 1000 evaluations\n")

# ============================================================================
# CHECKPOINT CONFIGURATION
# ============================================================================

# Set this to True to enable checkpointing (saves after each generation)
ENABLE_CHECKPOINTING = True

# Set this to True to resume from last checkpoint if available
RESUME_IF_AVAILABLE = True

# Checkpoint directory
checkpoint_dir = os.path.join(PROJECT_PATH, "optimization_checkpoints")
os.makedirs(checkpoint_dir, exist_ok=True)

# Run ID (unique identifier for this optimization run)
# Change this for different optimization runs
RUN_ID = "demo_run_001"  # or use timestamp: f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

checkpoint_path = os.path.join(checkpoint_dir, f"nsga3_{RUN_ID}_checkpoint.pkl")
results_path = os.path.join(checkpoint_dir, f"nsga3_{RUN_ID}_results.pkl")

print(f"Checkpoint configuration:")
print(f"  Checkpointing: {'ENABLED' if ENABLE_CHECKPOINTING else 'DISABLED'}")
print(f"  Resume: {'ENABLED' if RESUME_IF_AVAILABLE else 'DISABLED'}")
print(f"  Run ID: {RUN_ID}")
print(f"  Checkpoint file: {os.path.basename(checkpoint_path)}")

# ============================================================================
# CHECK FOR EXISTING CHECKPOINT
# ============================================================================

resume_from_checkpoint = False
starting_generation = 0
checkpoint_data = None

if RESUME_IF_AVAILABLE and os.path.exists(checkpoint_path):
    print(f"\n[FOUND] Existing checkpoint at: {checkpoint_path}")
    
    try:
        with open(checkpoint_path, 'rb') as f:
            checkpoint_data = pickle.load(f)
        
        starting_generation = checkpoint_data['generation']
        print(f"[OK] Checkpoint loaded successfully")
        print(f"     Last completed generation: {starting_generation}")
        print(f"     Evaluations completed: {checkpoint_data['n_eval']}")
        print(f"     Checkpoint timestamp: {checkpoint_data['timestamp']}")
        
        resume_from_checkpoint = True
        
    except Exception as e:
        print(f"[ERROR] Failed to load checkpoint: {e}")
        print(f"[INFO] Starting optimization from scratch")
        resume_from_checkpoint = False
else:
    if RESUME_IF_AVAILABLE:
        print(f"\n[INFO] No checkpoint found. Starting new optimization run.")
    else:
        print(f"\n[INFO] Resume disabled. Starting new optimization run.")

# ============================================================================
# CREATE OPTIMIZATION PROBLEM
# ============================================================================

problem = BreastCancerOptimizationProblem(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    device=device,
    batch_size=4,
    num_workers=2,
    patience=3,
    max_epochs=5 if not USE_FULL_DATASET else 50,
    pos_weight=n_benign/n_malignant,
    random_seed=42
)

# Generate reference directions for 4 objectives
ref_dirs = get_reference_directions("das-dennis", 4, n_partitions=3)

# ============================================================================
# CREATE OR RESTORE ALGORITHM
# ============================================================================

if resume_from_checkpoint:
    print("\n[RESUMING] Restoring algorithm state from checkpoint...")
    
    # Restore algorithm state
    algorithm = checkpoint_data['algorithm']
    
    # Calculate remaining generations
    total_generations = 3  # DEMO: 3, for production use 50
    remaining_generations = total_generations - starting_generation
    
    print(f"  Total generations: {total_generations}")
    print(f"  Completed: {starting_generation}")
    print(f"  Remaining: {remaining_generations}")
    
    if remaining_generations <= 0:
        print(f"\n[COMPLETE] Optimization already finished!")
        print(f"[INFO] Loading final results...")
        
        if os.path.exists(results_path):
            with open(results_path, 'rb') as f:
                res = pickle.load(f)
            print(f"[OK] Results loaded from: {results_path}")
        else:
            print(f"[ERROR] Results file not found!")
            res = None
        
        # Skip to visualization
        run_optimization = False
    else:
        run_optimization = True
        n_gen = remaining_generations
        
else:
    print("\n[STARTING] New optimization run...")
    
    # Create fresh algorithm
    algorithm = NSGA3(
        ref_dirs=ref_dirs,
        pop_size=5  # DEMO: 5 instead of 20
    )
    
    total_generations = 3  # DEMO: 3, for production use 50
    n_gen = total_generations
    run_optimization = True

# ============================================================================
# CUSTOM CALLBACK FOR CHECKPOINTING
# ============================================================================

class CheckpointCallback:
    """Save checkpoint after each generation."""
    
    def __init__(self, checkpoint_path, results_path, run_id):
        self.checkpoint_path = checkpoint_path
        self.results_path = results_path
        self.run_id = run_id
        self.generation_times = []
        self.last_time = time.time()
        
    def __call__(self, algorithm):
        # Track time
        current_time = time.time()
        gen_time = current_time - self.last_time
        self.generation_times.append(gen_time)
        self.last_time = current_time
        
        # Get current generation
        n_gen = algorithm.n_gen
        
        # Save checkpoint
        checkpoint_data = {
            'algorithm': algorithm,
            'generation': n_gen,
            'n_eval': algorithm.evaluator.n_eval,
            'timestamp': datetime.now().isoformat(),
            'run_id': self.run_id,
            'generation_times': self.generation_times,
        }
        
        with open(self.checkpoint_path, 'wb') as f:
            pickle.dump(checkpoint_data, f)
        
        # Print progress
        avg_time = sum(self.generation_times) / len(self.generation_times)
        print(f"  Gen {n_gen}: {algorithm.evaluator.n_eval} evals | "
              f"Time: {gen_time:.1f}s | Avg: {avg_time:.1f}s/gen | "
              f"Checkpoint saved")

# ============================================================================
# RUN OPTIMIZATION
# ============================================================================

if run_optimization:
    print(f"\nStarting optimization...\n")
    print(f"  Generations to run: {n_gen}")
    print(f"  Population size: {algorithm.pop_size}")
    print(f"  Expected evaluations: {algorithm.pop_size * n_gen}")
    
    if ENABLE_CHECKPOINTING:
        callback = CheckpointCallback(checkpoint_path, results_path, RUN_ID)
        print(f"  Checkpointing: ENABLED (saves after each generation)")
    else:
        callback = None
        print(f"  Checkpointing: DISABLED")
    
    print("\n" + "-"*80 + "\n")
    
    # Run optimization
    res = minimize(
        problem,
        algorithm,
        ('n_gen', n_gen),
        callback=callback,
        verbose=True,
        save_history=False
    )
    
    # Save final results
    print(f"\n[OK] Optimization complete!")
    print(f"  Total evaluations: {res.algorithm.n_eval}")
    print(f"  Pareto solutions: {len(res.F)}")
    
    # Save final results
    with open(results_path, 'wb') as f:
        pickle.dump(res, f)
    print(f"\n[SAVED] Final results to: {results_path}")
    
    # Clean up checkpoint (optimization finished)
    if os.path.exists(checkpoint_path):
        os.remove(checkpoint_path)
        print(f"[CLEANED] Removed checkpoint (optimization complete)")

# ============================================================================
# SUMMARY
# ============================================================================

print("\n" + "="*80)
if run_optimization:
    print("OPTIMIZATION COMPLETED SUCCESSFULLY")
else:
    print("OPTIMIZATION ALREADY COMPLETE")
print("="*80)

if res is not None:
    print(f"\nResults:")
    print(f"  Evaluations: {res.algorithm.n_eval}")
    print(f"  Pareto solutions: {len(res.F)}")
    print(f"  Results saved to: {os.path.basename(results_path)}")
    
    print(f"\nTo resume or restart:")
    print(f"  - Same run:  Keep RUN_ID = '{RUN_ID}' and RESUME_IF_AVAILABLE = True")
    print(f"  - New run:   Change RUN_ID to a new value")
    print(f"  - From scratch: Set RESUME_IF_AVAILABLE = False or delete checkpoint")
    
print("="*80 + "\n")

## 12. Visualize Pareto Front

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

print("\n" + "="*80)
print("PARETO FRONT SOLUTIONS")
print("="*80 + "\n")

# Display table
print(f"{'ID':<5} {'PR-AUC':>8} {'AUROC':>8} {'Brier':>8} {'Robust':>8}")
print("-" * 45)

for i, f in enumerate(res.F):
    pr_auc = -f[0]  # Convert back from minimization
    auroc = -f[1]
    brier = f[2]
    robust = f[3]
    
    print(f"{i:<5} {pr_auc:>8.4f} {auroc:>8.4f} {brier:>8.4f} {robust:>8.4f}")

# Visualize 2D projections
pr_auc = -res.F[:, 0]
auroc = -res.F[:, 1]
brier = res.F[:, 2]
robust = res.F[:, 3]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# PR-AUC vs AUROC
axes[0, 0].scatter(pr_auc, auroc, c='blue', s=100)
axes[0, 0].set_xlabel('PR-AUC')
axes[0, 0].set_ylabel('AUROC')
axes[0, 0].set_title('PR-AUC vs AUROC')
axes[0, 0].grid(True)

# PR-AUC vs Brier
axes[0, 1].scatter(pr_auc, brier, c='red', s=100)
axes[0, 1].set_xlabel('PR-AUC')
axes[0, 1].set_ylabel('Brier Score')
axes[0, 1].set_title('PR-AUC vs Brier')
axes[0, 1].grid(True)

# PR-AUC vs Robustness
axes[0, 2].scatter(pr_auc, robust, c='green', s=100)
axes[0, 2].set_xlabel('PR-AUC')
axes[0, 2].set_ylabel('Robustness Degradation')
axes[0, 2].set_title('PR-AUC vs Robustness')
axes[0, 2].grid(True)

# AUROC vs Brier
axes[1, 0].scatter(auroc, brier, c='purple', s=100)
axes[1, 0].set_xlabel('AUROC')
axes[1, 0].set_ylabel('Brier Score')
axes[1, 0].set_title('AUROC vs Brier')
axes[1, 0].grid(True)

# AUROC vs Robustness
axes[1, 1].scatter(auroc, robust, c='orange', s=100)
axes[1, 1].set_xlabel('AUROC')
axes[1, 1].set_ylabel('Robustness Degradation')
axes[1, 1].set_title('AUROC vs Robustness')
axes[1, 1].grid(True)

# Brier vs Robustness
axes[1, 2].scatter(brier, robust, c='brown', s=100)
axes[1, 2].set_xlabel('Brier Score')
axes[1, 2].set_ylabel('Robustness Degradation')
axes[1, 2].set_title('Brier vs Robustness')
axes[1, 2].grid(True)

plt.tight_layout()
plt.show()

print("\nNote: This is a DEMO with reduced scale.")
print("For production, use scripts/run_nsga3.py with full parameters.")

In [ ]:
# Display Pareto front
print("\nPareto Front Solutions:\n")
print(f"{'ID':<5} {'PR-AUC':>8} {'AUROC':>8} {'Brier':>8} {'Robust':>8}")
print("-" * 45)

for i, f in enumerate(res.F):
    pr_auc = -f[0]  # Convert back from minimization
    auroc = -f[1]
    brier = f[2]
    robust = f[3]
    
    print(f"{i:<5} {pr_auc:>8.4f} {auroc:>8.4f} {brier:>8.4f} {robust:>8.4f}")

print("\nNote: This is a DEMO with reduced scale.")
print("For production, use: pop_size=20, n_gen=50 in run_nsga3.py")

In [ ]:
# ============================================================================
# CHECKPOINT MANAGEMENT UTILITIES
# ============================================================================

print("="*80)
print("CHECKPOINT MANAGEMENT")
print("="*80)

# List all checkpoints
checkpoint_dir = os.path.join(PROJECT_PATH, "optimization_checkpoints")

if os.path.exists(checkpoint_dir):
    checkpoint_files = [f for f in os.listdir(checkpoint_dir) if f.endswith('.pkl')]
    
    if checkpoint_files:
        print(f"\nFound {len(checkpoint_files)} checkpoint files:\n")
        
        for f in sorted(checkpoint_files):
            filepath = os.path.join(checkpoint_dir, f)
            size_mb = os.path.getsize(filepath) / 1e6
            
            # Try to load and show info
            try:
                with open(filepath, 'rb') as file:
                    data = pickle.load(file)
                
                if 'generation' in data:  # It's a checkpoint
                    print(f"  CHECKPOINT: {f}")
                    print(f"    Generation: {data['generation']}")
                    print(f"    Evaluations: {data['n_eval']}")
                    print(f"    Timestamp: {data.get('timestamp', 'N/A')}")
                    print(f"    Size: {size_mb:.1f} MB")
                    
                    # Show timing info if available
                    if 'generation_times' in data and data['generation_times']:
                        avg_time = sum(data['generation_times']) / len(data['generation_times'])
                        print(f"    Avg time/gen: {avg_time:.1f}s")
                        
                elif hasattr(data, 'F'):  # It's results
                    print(f"  RESULTS: {f}")
                    print(f"    Pareto solutions: {len(data.F)}")
                    print(f"    Total evaluations: {data.algorithm.n_eval}")
                    print(f"    Size: {size_mb:.1f} MB")
                print()
                
            except Exception as e:
                print(f"  {f} (Size: {size_mb:.1f} MB) - Error loading: {e}\n")
    else:
        print("\nNo checkpoint files found.")
else:
    print("\nCheckpoint directory does not exist.")

print("="*80)
print("\nManagement Commands:")
print("  - To delete a checkpoint: Uncomment code below and set RUN_ID_TO_DELETE")
print("  - To start fresh: Set RESUME_IF_AVAILABLE = False in optimization cell")
print("  - To continue: Keep same RUN_ID and RESUME_IF_AVAILABLE = True")
print("="*80)

# ============================================================================
# DELETE CHECKPOINT (USE WITH CAUTION)
# ============================================================================

# Uncomment to delete a specific checkpoint and start fresh
# RUN_ID_TO_DELETE = "demo_run_001"
# checkpoint_to_delete = os.path.join(checkpoint_dir, f"nsga3_{RUN_ID_TO_DELETE}_checkpoint.pkl")
# if os.path.exists(checkpoint_to_delete):
#     os.remove(checkpoint_to_delete)
#     print(f"\nDeleted checkpoint: {checkpoint_to_delete}")
# else:
#     print(f"\nCheckpoint not found: {checkpoint_to_delete}")

## 13. Test INbreast Dataset Loading

Load INbreast dataset with the configured preprocessor for zero-shot evaluation.

In [ ]:
from src.datasets import INbreastDataset

# Load INbreast dataset with configured preprocessor
print("Loading INbreast dataset...")
print(f"Preprocessor: {'Adaptive' if USE_ADAPTIVE_PREPROCESSING else 'Standard'}\n")

inbreast_dataset = INbreastDataset(
    dicom_dir=INBREAST_DICOM_DIR,
    csv_file=INBREAST_CSV,
    preprocessor=preprocessor
)

print(f"[OK] INbreast dataset loaded: {len(inbreast_dataset)} samples")

# Test sample
img, label = inbreast_dataset[0]
print(f"\nSample shape: {img.shape}")
print(f"Sample label: {label.item()} ({'Malignant' if label.item() == 1 else 'Benign'})")

# Visualize
plt.figure(figsize=(8, 6))
plt.imshow(img.permute(1, 2, 0))
plt.title(f"INbreast Sample - {'Malignant' if label.item() == 1 else 'Benign'}")
plt.axis('off')
plt.show()

## 14. Zero-Shot Evaluation on INbreast

Transfer the trained model to INbreast **without any fine-tuning or threshold adjustment**.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from src.evaluation import aggregate_breast_level_predictions, compute_metrics

print("="*80)
print("ZERO-SHOT EVALUATION ON INBREAST")
print("="*80)

# Ensure model is in eval mode
model.eval()

# Evaluate on INbreast (zero-shot)
print("\n[1/2] Aggregating breast-level predictions...")
y_true_inbreast, y_probs_inbreast = aggregate_breast_level_predictions(
    model, inbreast_dataset, device
)

print(f"Number of breasts: {len(y_true_inbreast)}")
print(f"  Benign: {sum(y_true_inbreast == 0)}")
print(f"  Malignant: {sum(y_true_inbreast == 1)}")

# Compute metrics
print("\n[2/2] Computing metrics...")

# Find optimal threshold from VinDr validation set
# (Normally you'd compute this from validation predictions)
# For demo, using a default
optimal_threshold = 0.5

inbreast_metrics = compute_metrics(
    y_true_inbreast,
    y_probs_inbreast,
    threshold=optimal_threshold
)

# Display results
print("\n" + "="*80)
print("ZERO-SHOT RESULTS")
print("="*80)

print(f"\nThreshold-Independent Metrics:")
print(f"  PR-AUC:       {inbreast_metrics['pr_auc']:.4f}")
print(f"  AUROC:        {inbreast_metrics['auroc']:.4f}")
print(f"  Brier Score:  {inbreast_metrics['brier']:.4f}")

print(f"\nPrediction Distribution:")
print(f"  Min probability:  {y_probs_inbreast.min():.4f}")
print(f"  Max probability:  {y_probs_inbreast.max():.4f}")
print(f"  Spread:           {y_probs_inbreast.max() - y_probs_inbreast.min():.4f} ({(y_probs_inbreast.max() - y_probs_inbreast.min())*100:.1f}% of [0,1] range)")

print(f"\nThreshold-Dependent (t={optimal_threshold:.4f}):")
print(f"  Sensitivity:  {inbreast_metrics['sensitivity']:.4f}")
print(f"  Specificity:  {inbreast_metrics['specificity']:.4f}")

print(f"\nConfusion Matrix:")
print(f"  TN={inbreast_metrics['tn']:<4} FP={inbreast_metrics['fp']:<4}")
print(f"  FN={inbreast_metrics['fn']:<4} TP={inbreast_metrics['tp']:<4}")

# Check for domain shift
prob_spread = y_probs_inbreast.max() - y_probs_inbreast.min()
if prob_spread < 0.20:
    print("\n[WARNING] Severe domain shift detected!")
    print(f"  - Predictions collapsed to narrow range: {prob_spread*100:.1f}% of [0,1]")
    print(f"  - Model has minimal discriminative power on INbreast")
    print(f"  - Entropy adaptation may help (see Section 15-16)")
elif inbreast_metrics['auroc'] < 0.60:
    print("\n[WARNING] Poor generalization detected!")
    print(f"  - AUROC barely above random (0.5)")
    print(f"  - Domain shift likely present")

print("\n" + "="*80)

## 15. Entropy-Based Domain Adaptation

The zero-shot results above may show domain shift. Here we apply **entropy-based domain adaptation** to align INbreast intensity distribution with VinDr-Mammo training statistics.

In [ ]:
# This section demonstrates entropy-based domain adaptation
# To use adaptation:
#   1. Set USE_ADAPTIVE_PREPROCESSING = True in Section 3
#   2. Run Section 16 to compute/load entropy statistics
#   3. Re-run from Section 4 onwards with adapted preprocessor

print("Current Configuration:")
print(f"  Adaptive Preprocessing: {USE_ADAPTIVE_PREPROCESSING}")
print(f"  Entropy Stats Available: {ENTROPY_STATS_AVAILABLE}")

if USE_ADAPTIVE_PREPROCESSING and ENTROPY_STATS_AVAILABLE:
    print("\n[OK] Adaptive preprocessing is ENABLED")
    print("     All evaluations above used entropy-adapted images")
else:
    print("\n[INFO] Adaptive preprocessing is DISABLED")
    print("       Run Section 16 to enable domain adaptation")

## 16. Entropy Statistics & Comparative Analysis

If adaptive preprocessing is disabled above, this section allows you to compare:
1. **Baseline** - Standard preprocessing (as evaluated in Section 15)
2. **Adapted** - Entropy-based adaptive preprocessing

This comparison demonstrates the impact of domain adaptation on zero-shot performance.

In [ ]:
from src.domain_adaptation import compute_dataset_entropy_stats, EntropyStatistics
from src.cache_manager import EntropyCache

print("="*80)
print("ENTROPY STATISTICS COMPUTATION")
print("="*80)

# Check cache
cache = EntropyCache(cache_dir=os.path.join(PROJECT_PATH, "cache"))

if cache.exists("vindr_train"):
    print("\n[OK] Loading cached entropy statistics...")
    entropy_stats = cache.load("vindr_train")
    
    print(f"\nEntropy Statistics (from cache):")
    print(f"  Mean:     {entropy_stats.mean:.4f} bits")
    print(f"  Std:      {entropy_stats.std:.4f} bits")
    print(f"  Min:      {entropy_stats.min_entropy:.4f} bits")
    print(f"  Max:      {entropy_stats.max_entropy:.4f} bits")
    print(f"  Samples:  {entropy_stats.n_samples}")
    if hasattr(entropy_stats, 'computed_date'):
        print(f"  Computed: {entropy_stats.computed_date}")
    print(f"\nTarget range for adaptation: [{entropy_stats.mean - entropy_stats.std:.4f}, {entropy_stats.mean + entropy_stats.std:.4f}] bits")
else:
    print("\n[INFO] Cache not found. Computing entropy statistics...")
    print("This may take 1-2 hours for full VinDr-Mammo training set.\n")
    
    from src.adaptive_preprocessing import AdaptiveMammographyPreprocessor
    
    # Create adaptive preprocessor (no adaptation, just for grayscale access)
    adaptive_preprocessor = AdaptiveMammographyPreprocessor(
        apply_adaptation=False
    )
    
    print("Computing entropy statistics on training set...")
    print("Progress will be shown below:\n")
    
    # Note: This requires train_dataset to be defined
    # For demo purposes, computing on full dataset (should be training only)
    entropy_stats = compute_dataset_entropy_stats(
        dataset=dataset,
        preprocessor=adaptive_preprocessor,
        save_path=None,
        verbose=True
    )
    
    # Set dataset name
    entropy_stats.dataset_name = "VinDr-Mammo-Train"
    
    # Save to cache
    print(f"\nSaving statistics to cache...")
    cache.save(entropy_stats, "vindr_train")
    
    print(f"\n[OK] Computation complete!")
    print(f"\nEntropy Statistics:")
    print(f"  Mean:     {entropy_stats.mean:.4f} bits")
    print(f"  Std:      {entropy_stats.std:.4f} bits")
    print(f"  Min:      {entropy_stats.min_entropy:.4f} bits")
    print(f"  Max:      {entropy_stats.max_entropy:.4f} bits")
    print(f"  Samples:  {entropy_stats.n_samples}")
    print(f"\nTarget range for adaptation: [{entropy_stats.mean - entropy_stats.std:.4f}, {entropy_stats.mean + entropy_stats.std:.4f}] bits")
    print(f"\nCache saved to: {cache.get_cache_path('vindr_train')}")

print("\n" + "="*80)
print("\nTo enable adaptive preprocessing:")
print("  1. Set USE_ADAPTIVE_PREPROCESSING = True in Section 3")
print("  2. Restart notebook runtime (or re-run from Section 3)")
print("  3. All subsequent sections will use adapted images")
print("\n" + "="*80)

## 17. Final Summary & Usage Notes

### Entropy-Based Domain Adaptation Implementation

**Configuration:**
- Set `USE_ADAPTIVE_PREPROCESSING = True` in Section 3 to enable adaptation
- Preprocessing mode is automatically applied throughout the notebook
- All datasets (VinDr-Mammo, INbreast) use the selected preprocessor

**What Adaptive Preprocessing Does:**
- Computes Shannon entropy statistics from VinDr-Mammo training set (mean, std)
- Caches statistics for reuse (`cache/entropy_stats_vindr_train.json`)
- Applied iterative entropy transformation to align images:
  - Square image if entropy too low (increases contrast)
  - Square root image if entropy too high (decreases contrast)
  - Iterate until entropy reaches target range [mean-std, mean+std]
- No model retraining required!

**Key Files:**
- `src/domain_adaptation.py` - Entropy calculation and adaptive transform
- `src/adaptive_preprocessing.py` - Preprocessing wrapper with adaptation
- `src/cache_manager.py` - Entropy statistics caching
- `src/datasets.py` - Factory functions for adapted datasets
- `scripts/compute_entropy_stats.py` - One-time entropy computation
- `scripts/evaluate_zeroshot_adapted.py` - Multi-scenario evaluation

**To Compute Entropy Cache (One-Time):**

```bash
cd /content/drive/MyDrive/breast_cancer_detection
python scripts/compute_entropy_stats.py
```

**Expected Improvements with Adaptive Preprocessing:**
- INbreast prediction spread: >50% of [0,1] range (vs baseline ~5-10%)
- AUROC improvement: +0.10 to +0.20 (from 0.54 to 0.64-0.74)
- Better calibration: Lower Brier score
- Mitigates domain shift without retraining

**Next Steps:**
- Run full-scale NSGA-III optimization with adaptive preprocessing
- Evaluate all Pareto solutions on adapted INbreast
- Compare domain shift mitigation across different hyperparameter configurations